# Fake E-Commerce Review Detection — Model Training

Detecting and removing fake reviews helps maintain a fair marketplace, improves customer confidence, and ensures that product ratings accurately reflect real user experiences. It also protects platforms from manipulation, supports regulatory compliance, and enhances overall data quality for analytics and recommendation systems, making fake review detection a critical application of machine learning in e-commerce and digital platforms.


## 1. Import Libraries

In [11]:
import pandas as pd

## 2. Load Dataset

In [12]:
df = pd.read_csv('fake reviews dataset.csv')

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

## 3. Feature Engineering & Train/Test Split

In [14]:
# Target encoding: CG (fake) = 1, OR (genuine) = 0
df['label'] = df['label'].map({'CG': 1, 'OR': 0})

# Features and target (text, category, rating only)
X = df[['text_', 'category', 'rating']]
y = df['label']

In [15]:
# Test Train Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4. Text Cleaning

In [16]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()

In [17]:
def clean_text(text):

    # lowercase
    text = text.lower()

    # remove punctuation & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenize
    words = text.split()

    # remove stopwords + stemming
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

## 5. Build Pipeline (TF-IDF Bigrams + LinearSVC)

In [18]:
# TF-IDF with bigrams, using clean_text as preprocessor
tfidf_bigram_clean = TfidfVectorizer(
    preprocessor=clean_text,
    ngram_range=(1, 2),
    max_features=10000
)

# Column transformer: text + category (OHE) + rating (passthrough)
preprocessor = ColumnTransformer(
    transformers=[
        ('text', tfidf_bigram_clean, 'text_'),
        ('cat',  OneHotEncoder(handle_unknown='ignore'), ['category']),
        ('num',  'passthrough', ['rating'])
    ]
)

# Final pipeline
pipeline_clean_bigram_svm = Pipeline([
    ('preprocess', preprocessor),
    ('model', LinearSVC())
])

## 6. Train & Evaluate

In [ ]:
# Train the model
pipeline_clean_bigram_svm.fit(X_train, y_train)

# Evaluate on test set
y_pred = pipeline_clean_bigram_svm.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['OR (Genuine)', 'CG (Fake)']))

## 7. Save Model

In [ ]:
import pickle

# Save the trained pipeline as model.pkl for use in the Flask web app
model_path = 'model.pkl'

# with open(model_path, 'wb') as f:
#     pickle.dump(pipeline_clean_bigram_svm, f)


import joblib

joblib.dump(pipeline_clean_bigram_svm, "model.pkl")

print(f"Model saved successfully to '{model_path}'")


Model saved successfully to 'model.pkl'
